In [74]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

In [75]:
df = pd.read_csv('data/train.csv')
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [76]:
df = df.pipe(lambda d: d.rename(columns=lambda c: str(c).lower().strip().replace(' ', '_')))
df.head()

,passengerid,survived,pclass,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [77]:
# EDA
target_col = 'survived'

if 'passengerid' in df.columns:
    df.drop(columns=['passengerid'], inplace=True)

if 'name' in df.columns:
    df.drop(columns=['name'], inplace=True)

df['sex'] = df['sex'].str.lower().map({'male':1,'female':0})

df.head()

,survived,pclass,sex,age,sibsp,parch,ticket,fare,cabin,embarked
0,0,3,1,22.0,1,0,A/5 21171,7.2500,NaN,S
1,1,1,0,38.0,1,0,PC 17599,71.2833,C85,C
2,1,3,0,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,1,1,0,35.0,1,0,113803,53.1000,C123,S
4,0,3,1,35.0,0,0,373450,8.0500,NaN,S


In [78]:
# Missing age values
df['age'] = df['age'].fillna(df['age'].median())

In [ ]:
# Missing values in cabin column
print(df['cabin'].isna().sum() / len(df) * 100)

# Drop cabin column as it has more than 70% missing values
df.drop(columns=['cabin'], inplace=True)

In [80]:
print(df['ticket'].nunique(), df['ticket'].isna().sum(), len(df))

# Ticket feature has high ratio (22%) of duplicate values (unique=681).
df.drop(columns=['ticket'], inplace=True)

681 0 891


In [81]:
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked
0,0,3,1,22.0,1,0,7.2500,S
1,1,1,0,38.0,1,0,71.2833,C
2,1,3,0,26.0,0,0,7.9250,S
3,1,1,0,35.0,1,0,53.1000,S
4,0,3,1,35.0,0,0,8.0500,S


In [82]:
df['embarked'].isnull().sum()

np.int64(2)

In [83]:
df['embarked'].fillna(df['embarked'].mode()[0], inplace=True)

C:\Users\SubudhiK\AppData\Local\Temp\ipykernel_19008\1964997694.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['embarked'].fillna(df['embarked'].mode()[0], inplace=True)


In [84]:
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

encoded = encoder.fit_transform(df[['embarked']])

encoded_df = pd.DataFrame(
    encoded,
    columns=encoder.get_feature_names_out(['embarked']),
    index=df.index
)

df = pd.concat([df.drop(columns=['embarked']), encoded_df], axis=1)

df.head()


,survived,pclass,sex,age,sibsp,parch,fare,embarked_C,embarked_Q,embarked_S
0,0,3,1,22.0,1,0,7.2500,0.0,0.0,1.0
1,1,1,0,38.0,1,0,71.2833,1.0,0.0,0.0
2,1,3,0,26.0,0,0,7.9250,0.0,0.0,1.0
3,1,1,0,35.0,1,0,53.1000,0.0,0.0,1.0
4,0,3,1,35.0,0,0,8.0500,0.0,0.0,1.0


In [85]:
scaler = StandardScaler()

scaled_values = scaler.fit_transform(
    df[['age', 'fare', 'pclass', 'sibsp', 'parch']]
)

scaled_df = pd.DataFrame(
    scaled_values,
    columns=['age', 'fare', 'pclass', 'sibsp', 'parch'],
    index=df.index
)

df[['age', 'fare', 'pclass', 'sibsp', 'parch']] = scaled_df

df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked_C,embarked_Q,embarked_S
0,0,0.827377,1,-0.565736,0.432793,-0.473674,-0.502445,0.0,0.0,1.0
1,1,-1.566107,0,0.663861,0.432793,-0.473674,0.786845,1.0,0.0,0.0
2,1,0.827377,0,-0.258337,-0.474545,-0.473674,-0.488854,0.0,0.0,1.0
3,1,-1.566107,0,0.433312,0.432793,-0.473674,0.420730,0.0,0.0,1.0
4,0,0.827377,1,0.433312,-0.474545,-0.473674,-0.486337,0.0,0.0,1.0


In [86]:

X = df.drop(columns=[target_col])
y = df[target_col]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [87]:
# Logistic Regression model
from sklearn.linear_model import LogisticRegression

logreg = LogisticRegression()
logreg.fit(X_train, y_train)
y_pred = logreg.predict(X_test)
acc_logreg = round(logreg.score(X_test, y_test) * 100, 2)
acc_logreg

81.01

In [88]:
# Support Vector Machines
from sklearn.svm import SVC

svc = SVC(C=1.0, kernel='rbf', gamma='scale')
svc.fit(X_train, y_train)
y_pred = svc.predict(X_test)
acc_svc = round(svc.score(X_test, y_test) * 100, 2)
acc_svc

81.56

In [89]:
# knn model
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors = 3)
knn.fit(X_train, y_train)
y_pred = knn.predict(X_test)
acc_knn = round(knn.score(X_test, y_test) * 100, 2)
acc_knn

81.01

In [90]:
# Gaussian Naive Bayes
from sklearn.naive_bayes import GaussianNB

gaussian = GaussianNB()
gaussian.fit(X_train, y_train)
y_pred = gaussian.predict(X_test)
acc_gaussian = round(gaussian.score(X_test, y_test) * 100, 2)
acc_gaussian

77.09

In [91]:
# Perceptron
from sklearn.linear_model import Perceptron

perceptron = Perceptron(max_iter=1000, tol=1e-3, random_state=42)
perceptron.fit(X_train, y_train)
y_pred = perceptron.predict(X_test)
acc_perceptron = round(perceptron.score(X_test, y_test) * 100, 2)
acc_perceptron

74.86

In [92]:
# Linear SVC
from sklearn.svm import LinearSVC

linear_svc = LinearSVC(max_iter=1000, tol=1e-3, random_state=42)
linear_svc.fit(X_train, y_train)
y_pred = linear_svc.predict(X_test)
acc_linear_svc = round(linear_svc.score(X_test, y_test) * 100, 2)
acc_linear_svc

78.77

In [93]:
# Stochastic Gradient Descent
from sklearn.linear_model import SGDClassifier

sgd = SGDClassifier(max_iter=1000, tol=1e-3, random_state=42)
sgd.fit(X_train, y_train)
y_pred = sgd.predict(X_test)
acc_sgd = round(sgd.score(X_test, y_test) * 100, 2)
acc_sgd

77.65

In [94]:
# Decision Tree
from sklearn.tree import DecisionTreeClassifier

decision_tree = DecisionTreeClassifier(random_state=42, ccp_alpha=0.01)
decision_tree.fit(X_train, y_train)
y_pred = decision_tree.predict(X_test)
acc_decision_tree = round(decision_tree.score(X_test, y_test) * 100, 2)
acc_decision_tree

79.89

In [95]:
# Random Forest
from sklearn.ensemble import RandomForestClassifier

random_forest = RandomForestClassifier(n_estimators=100, random_state=42, min_samples_leaf=1, max_features=0.01)
random_forest.fit(X_train, y_train)
y_pred = random_forest.predict(X_test)
random_forest.score(X_train, y_train)
acc_random_forest = round(random_forest.score(X_test, y_test) * 100, 2)
acc_random_forest

81.01

In [98]:
models = pd.DataFrame({
    'Model': ['Support Vector Machines', 'KNN', 'Logistic Regression', 
              'Random Forest', 'Naive Bayes', 'Perceptron', 
              'Stochastic Gradient Decent', 'Linear SVC', 
              'Decision Tree'],
    'Score': [acc_svc, acc_knn, acc_logreg, 
              acc_random_forest, acc_gaussian, acc_perceptron, 
              acc_sgd, acc_linear_svc, acc_decision_tree]})
models.sort_values(by='Score', ascending=False)

,Model,Score
0,Support Vector Machines,81.56
1,KNN,81.01
2,Logistic Regression,81.01
3,Random Forest,81.01
8,Decision Tree,79.89
7,Linear SVC,78.77
6,Stochastic Gradient Decent,77.65
4,Naive Bayes,77.09
5,Perceptron,74.86


In [ ]:
# ANN model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# Initialize the ANN
ann = Sequential()
es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=10, restore_best_weights=True)

ann.add(Dense(units=32, activation='relu', input_shape=(X_train.shape[1],)))
ann.add(Dropout(0.2))
ann.add(Dense(units=16, activation='relu'))
ann.add(Dropout(0.2))
ann.add(Dense(units=8, activation='relu'))
ann.add(Dropout(0.3))
ann.add(Dense(units=1, activation='sigmoid'))
ann.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
history = ann.fit(X_train, y_train, batch_size=32, epochs=100, validation_split=0.2, callbacks=[es])
loss, acc_ann = ann.evaluate(X_test, y_test)
acc_ann = round(acc_ann * 100, 2)
acc_ann

Epoch 1/100


c:\Users\SubudhiK\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


18/18 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.6151 - loss: 0.6578 - val_accuracy: 0.6503 - val_loss: 0.6226
Epoch 2/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6134 - loss: 0.6390 - val_accuracy: 0.6503 - val_loss: 0.5995
Epoch 3/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6169 - loss: 0.6275 - val_accuracy: 0.6503 - val_loss: 0.5826
Epoch 4/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6257 - loss: 0.6204 - val_accuracy: 0.6573 - val_loss: 0.5689
Epoch 5/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6239 - loss: 0.6116 - val_accuracy: 0.6853 - val_loss: 0.5565
Epoch 6/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6467 - loss: 0.6003 - val_accuracy: 0.6923 - val_loss: 0.5479
Epoch 7/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6538 - loss: 0.5989 - val_accuracy: 0.7273 - val_loss: 0.5389
Epoch 8/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6837 - loss: 0.5843 - val_accuracy: 0.7273 - val_loss: 0.

83.24

In [105]:
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score

y_pred_prob = ann.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_pred_prob))

6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
[[95 10]
 [20 54]]
              precision    recall  f1-score   support

           0       0.83      0.90      0.86       105
           1       0.84      0.73      0.78        74

    accuracy                           0.83       179
   macro avg       0.83      0.82      0.82       179
weighted avg       0.83      0.83      0.83       179

ROC-AUC: 0.8903474903474903


In [107]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from scikeras.wrappers import KerasClassifier
import numpy as np

def create_model():
    ann = Sequential()
    ann.add(Dense(32, activation='relu', input_shape=(X.shape[1],)))
    ann.add(Dropout(0.2))
    ann.add(Dense(16, activation='relu'))
    ann.add(Dropout(0.2))
    ann.add(Dense(8, activation='relu'))
    ann.add(Dropout(0.3))
    ann.add(Dense(1, activation='sigmoid'))
    ann.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return ann

model = KerasClassifier(model=create_model, epochs=80, batch_size=32, verbose=0)

kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, X, y, cv=kfold, scoring='accuracy')

print("CV Accuracy scores:", scores)
print("Mean CV Accuracy:", scores.mean())
print("Std Dev:", scores.std())

c:\Users\SubudhiK\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
c:\Users\SubudhiK\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


c:\Users\SubudhiK\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


c:\Users\SubudhiK\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
c:\Users\SubudhiK\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


CV Accuracy scores: [0.84357542 0.80337079 0.80898876 0.8258427  0.8258427 ]
Mean CV Accuracy: 0.8215240725629277
Std Dev: 0.014214349517335101
